In [ ]:
!mkdir face-recognition-attendance
%cd face-recognition-attendance

mkdir: cannot create directory ‘face-recognition-attendance’: File exists
/content/face-recognition-attendance


In [ ]:
!pwd

/content/face-recognition-attendance


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil
import os

destination_directory = "/content/CLASS-ATTENDANCE/"

# Create the destination directory if it doesn't exist
os.makedirs(destination_directory, exist_ok=True)

try:
    shutil.copy("/content/drive/MyDrive/Colab Notebooks/FaceAttendance_final.ipynb", destination_directory)
    print(f"Successfully copied FaceAttendance_final.ipynb to {destination_directory}")
except FileNotFoundError:
    print("Error: Source file not found. Please check the path to 'FaceAttendance_final.ipynb' in your Google Drive.")
except Exception as e:
    print(f"An error occurred while copying the file: {e}")

Successfully copied FaceAttendance_final.ipynb to /content/CLASS-ATTENDANCE/


In [ ]:
# Install required libraries
!pip install face_recognition
!pip install opencv-contrib-python
!pip install openpyxl

In [ ]:
import cv2
import face_recognition
import os

dataset_path = "/content/drive/MyDrive/face_dataset"
video_folder = "/content/drive/MyDrive/FaceAttendance (1)/dataset"

os.makedirs(dataset_path, exist_ok=True)
videos = os.listdir(video_folder)

for video_file in videos:
    video_path = os.path.join(video_folder, video_file)
    raw_name = os.path.splitext(video_file)[0]

    name_parts = raw_name.split(' - ')
    if len(name_parts) > 1:
        person_name_for_display = name_parts[-1].strip()
    else:
        person_name_for_display = raw_name.strip()

    folder_name = person_name_for_display.title().replace(' ', '_')
    student_folder = os.path.join(dataset_path, folder_name)
    os.makedirs(student_folder, exist_ok=True)
    if len(os.listdir(student_folder)) > 15:
        print(f"{person_name_for_display.lower()} already done")
        continue

    cap = cv2.VideoCapture(video_path)

    frame_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = 15
    saved_count = 0
    for frame_number in range(0, frame_total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        faces = face_recognition.face_locations(rgb, model="hog")
        for top, right, bottom, left in faces:
            face = frame[top:bottom, left:right]
            filename = os.path.join(
                student_folder,
                f"{folder_name}_{saved_count}.jpg"
            )

            cv2.imwrite(filename, face)
            saved_count += 1

    cap.release()
    print(f"{person_name_for_display.lower()} {saved_count} faces")

parth raj 12 faces
pulkit raj 11 faces
ishaan chakraborty already done
manasi mohanty already done
aryan singh(1) 9 faces
aryan singh 9 faces
aditi jain 0 faces
manimoy karmakar 9 faces
sampriti ray 15 faces
vedansh parashar 1 faces
aditya anand 8 faces
keshab agarwal 7 faces
evelyn johnson already done
aditya anand 6 faces
ananway goswami 10 faces
saswat pradhan 15 faces
lucky jha already done
sanskriti vashishtha 13 faces
shankhadeep paul 6 faces
sourav sanu 15 faces
rajashree das 15 faces
pbidyusmita patro.facevideo 14 faces
srijan basu 12 faces
aditya pathak 15 faces
sukrit dutta already done
pranjal sinha already done
ritesh patel 9 faces
shivam singh already done
krishna rakshit already done
stuti shah already done
ramya panda already done
sreeraj saha already done
shivangi sinha 11 faces
abhinav kumar 11 faces
pallab ghosh already done
tanmay already done
aayush hore already done
yogesh sangwan 12 faces
manshi mehul already done
aayansh jaiswal 9 faces
jeet naskar 14 faces
md ta

In [2]:
print(video_folder)
!ls -R "/content/drive/MyDrive/FaceAttendance (1)/dataset"

NameError: name 'video_folder' is not defined

In [3]:
# Generate face embeddings from dataset images

import face_recognition
import os
import pickle

dataset_path = "/content/drive/MyDrive/face_dataset"

known_embeddings = []
known_names = []
for person_name in os.listdir(dataset_path):
    person_folder = os.path.join(dataset_path, person_name)
    if not os.path.isdir(person_folder):
        continue
    print("Processing:", person_name)
    for image_name in os.listdir(person_folder):
        image_path = os.path.join(person_folder, image_name)
        image = face_recognition.load_image_file(image_path)
        encodings = face_recognition.face_encodings(image)
        if len(encodings) > 0:
            known_embeddings.append(encodings[0])
            known_names.append(person_name)


data = {
    "embeddings": known_embeddings,
    "names": known_names
}

with open("/content/drive/MyDrive/FaceAttendance/embedding_database.pkl", "wb") as f:
    pickle.dump(data, f)

print("Embedding database created successfully!")

ModuleNotFoundError: No module named 'face_recognition'

In [ ]:
# Create student database Excel — run once

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
import os
dataset_path = "/content/drive/MyDrive/face_dataset"
excel_db_path = "/content/drive/MyDrive/FaceAttendance/student_database.xlsx"
student_names = sorted([
    d for d in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, d))
])

print(f"Found {len(student_names)} students:")
for s in student_names:
    print(" ", s)
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Students"

header_fill = PatternFill("solid", fgColor="1F4E79")
header_font = Font(bold=True, color="FFFFFF", size=12)

for col, h in enumerate(["S.No", "Student ID", "Name", "Folder Name"], 1):
    cell = ws.cell(row=1, column=col, value=h)
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center")

for i, folder_name in enumerate(student_names, 1):
    display_name = folder_name.replace("_", " ").title()
    ws.append([i, f"STU{i:03d}", display_name, folder_name])

ws.column_dimensions['A'].width = 8
ws.column_dimensions['B'].width = 12
ws.column_dimensions['C'].width = 28
ws.column_dimensions['D'].width = 28

wb.save(excel_db_path)
print(f"\n✅ Student database saved → {excel_db_path}")

In [ ]:
import face_recognition
import cv2
import pickle
import os
import numpy as np
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from datetime import datetime
from google.colab.patches import cv2_imshow

# Load embedding database
with open("/content/drive/MyDrive/FaceAttendance/embedding_database.pkl", "rb") as f:
    data = pickle.load(f)

known_embeddings = data["embeddings"]
known_names = data["names"]

print("Total embeddings loaded:", len(known_embeddings))

image_folder = "/content/drive/MyDrive/FaceAttendance/testing_data"

present_students = set()
THRESHOLD = 0.5

if not os.path.exists(image_folder):
    print(f"Error: Folder not found → {image_folder}")
else:
    for image_name in os.listdir(image_folder):
        image_path = os.path.join(image_folder, image_name)
        print("\nProcessing image:", image_name)

        image = cv2.imread(image_path)
        if image is None:
            print("Skipping invalid image")
            continue

        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        face_locations = face_recognition.face_locations(rgb_image, model="cnn")
        face_encodings = face_recognition.face_encodings(rgb_image, face_locations)

        print("Faces detected:", len(face_locations))

        for (top, right, bottom, left), encoding in zip(face_locations, face_encodings):

            face_distances = face_recognition.face_distance(known_embeddings, encoding)

            if len(face_distances) == 0:
                print("No embeddings available!")
                continue

            best_match_index = np.argmin(face_distances)
            distance = face_distances[best_match_index]

            if distance < THRESHOLD:
                name = known_names[best_match_index]
                present_students.add(name)
            else:
                name = "Unknown"

            print("Predicted:", name, "| Distance:", round(distance, 3))

            cv2.rectangle(image, (left, top), (right, bottom), (0,255,0), 2)
            cv2.putText(image, name, (left, top - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

        cv2_imshow(image)